# 여러 글자를 한 번에 생성 → 칸별로 잘라 획 분석

**여러 글자를 따로 생성해서 합치는 방식이 아닙니다.** 학습 기준 글자들을 한 장에 배치하고,
그 전체 이미지를 Stable Diffusion img2img에 넣어 한 번에 역확산합니다.
생성한 전체 이미지를 저장한 뒤, 미리 고정한 칸 좌표로 잘라 기존 Topo와 원본 점수로 분석합니다.
OCR 성공 여부로 자르거나 결과를 제외하지 않습니다. 빈 칸·합쳐진 글자·다른 글자도 남깁니다.

기본 시험: 같은 기준 목록의 앞 1/4/9개, 전체 576×576, strength 0.65/0.85, seed 100.
공통으로 포함된 글자를 비교할 수 있습니다. 서로 다른 글자들을 전체 평균만으로 비교하지 마세요.
글자가 많아질수록 글자당 픽셀도 줄어듭니다. 개수 효과와 해상도 효과를 분리해 보려면
2번 셀에서 FIXED_CELL_PIXELS=192로 설정하세요(전체 크기가 달라져 계산량도 달라집니다).
글자 수가 많으면 오류가 더 생길지는 실제 결과로 확인해야 합니다.

1. 준비·Drive → 2. 기존 학습 ZIP/CSV 자동 준비 → 3. 학습 글자 여러 개 선택
4. 전체 이미지 공동 생성·원본 보기 → 5. 잘라낸 글자별 추출·점수·감점 지도

GPU 런타임에서 실행하세요. 5번 셀은 기존 평가를 실행하며 역확산 guidance는 적용하지 않습니다.
현재 원본 점수는 옅은 생성 배경에도 민감하므로 감점 지도와 원본을 함께 확인하세요.

In [ ]:
from pathlib import Path
import subprocess, sys, tempfile, json
root = Path(tempfile.mkdtemp(prefix='hsqm_diffusion_score_', dir='/content'))
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main',
    'https://github.com/yechan25/hsqm.git', str(root)], check=True)
commit = subprocess.check_output(['git', '-C', str(root), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'diffusers==0.35.1', 'transformers==4.49.0', 'accelerate', 'safetensors',
    'timm', 'pandas', 'numpy', 'tqdm', 'matplotlib', 'pillow', 'scipy',
    'scikit-image', 'pyyaml', 'fonttools', 'ipywidgets'], check=True)
sys.path.insert(0, str(root))
for key in list(sys.modules):
    if key.split('.')[0] in ('stroke_model', 'font_test', 'diffusion_model'):
        del sys.modules[key]
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Colab에서 GPU 런타임으로 바꿔주세요.')
print('GPU:', torch.cuda.get_device_name(0))
print('실행 GitHub 커밋:', commit)
from google.colab import drive, output
drive.mount('/content/drive')
output.enable_custom_widget_manager()

In [ ]:
from stroke_model.convert_reference_csv import existing_path
from diffusion_model.score_pilot import training_catalog, generate_pilot, score_pilot, load_catalog_reference
from diffusion_model.pilot_data import prepare_pilot_data
from diffusion_model.multiglyph_pilot import generate_multiglyph_pilot
from IPython.display import display
import ipywidgets as widgets
HSQM = existing_path('/content/drive/Shareddrives/2026 자율연구/HSQM')
CHECKPOINT = None  # 폰트 실험에서 사용한 best.pt의 전체 경로를 넣으면 그 모델 사용
DATA_ROOT = HSQM/'dataset/automated_train_dataset'  # 기존 학습 ZIP/TAR가 있는 폴더
CSV_PATH = None  # None: 체크포인트 옆에 저장된 학습 CSV를 자동 사용
TRAIN_MANIFEST = None  # 기존 세션의 유효한 목록이 있을 때만 선택적으로 지정. 없어도 자동 재생성
REFERENCE_ONLY = True  # 시험에는 기준 글자/기준 획만 사용. 전체 학습 NPZ 변환 생략
SCORE_CONFIG = {}  # 폰트 시험에서 값을 바꿨다면 동일한 설정 입력
SOURCE_CONFIG = {}
GLYPH_COUNTS = [1, 4, 9]
CANVAS_SIZE = 576
FIXED_CELL_PIXELS = None  # None: 전체 크기 고정. 192: 글자당 크기 고정 비교
STRENGTHS = [.65, .85]
SEEDS = [100]
STEPS = 30
MODEL_ID = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
if CHECKPOINT is None:
    candidates = sorted((HSQM/'v2_topology_runs').glob('*/best.pt'))
    if not candidates:
        raise FileNotFoundError('CHECKPOINT에 기존 Topo best.pt를 지정하세요.')
    CHECKPOINT = candidates[-1]
CHECKPOINT = existing_path(CHECKPOINT)
print('사용할 획 모델:', CHECKPOINT, flush=True)
preparation = prepare_pilot_data(CHECKPOINT, DATA_ROOT, csv_path=CSV_PATH, manifest=TRAIN_MANIFEST,
    reference_only=REFERENCE_ONLY)
TRAIN_MANIFEST = Path(preparation['manifest'])
print('데이터 준비 완료. 3번 셀에서 기준 글자를 선택하세요.', flush=True)

In [ ]:
import io, math
import numpy as np
from PIL import Image, ImageDraw
catalog = training_catalog(TRAIN_MANIFEST)
by_row = {item['manifest_row']: item for item in catalog}
needed = max(GLYPH_COUNTS)
if len(catalog) < needed:
    raise ValueError(f'사용 가능한 학습 기준은 {len(catalog)}개입니다. GLYPH_COUNTS를 줄이세요.')
reference_select = widgets.SelectMultiple(
    options=[(f"train row {item['manifest_row']} | {item['char_id'] or '(그림 확인)'}", item['manifest_row']) for item in catalog],
    value=tuple(item['manifest_row'] for item in catalog[:needed]), rows=12,
    description='여러 글자:', layout=widgets.Layout(width='98%'))
preview = widgets.Image(format='png', width=500, height=500)
def update_preview(change=None):
    selected = list(reference_select.value)[:needed]
    side = math.ceil(math.sqrt(needed))
    sheet = Image.new('RGB', (160*side, 180*side), 'white')
    draw = ImageDraw.Draw(sheet)
    for index, row in enumerate(selected):
        reference,_ = load_catalog_reference(by_row[row])
        glyph = Image.fromarray(np.uint8((1-reference[0])*255)).convert('RGB')
        x, y = (index%side)*160, (index//side)*180
        sheet.paste(glyph.resize((160,160)), (x,y))
        draw.text((x+4,y+160), f'slot {index+1} | row {row}', fill='black')
    buffer=io.BytesIO(); sheet.save(buffer,format='PNG'); preview.value=buffer.getvalue()
reference_select.observe(update_preview,names='value'); update_preview()
display(widgets.VBox([reference_select,preview]))
print(f'최소 {needed}개를 선택하세요. 각 개수 조건은 선택 목록의 앞부분을 사용합니다.')
print('문자명이 없는 경우 그림으로 확인하세요. 서로 다른 글자를 비교하려면 같은 글자의 폰트 변형을 중복 선택하지 마세요.')

In [ ]:
from datetime import datetime
from huggingface_hub import model_info
MODEL_REVISION = model_info(MODEL_ID).sha
OUTPUT_DIR = HSQM/'multiglyph_diffusion_pilots'/datetime.now().strftime('%Y%m%d_%H%M%S_%f')
generation = generate_multiglyph_pilot(TRAIN_MANIFEST, list(reference_select.value), OUTPUT_DIR,
    glyph_counts=GLYPH_COUNTS, canvas_size=CANVAS_SIZE, cell_pixels=FIXED_CELL_PIXELS,
    strengths=STRENGTHS, seeds=SEEDS, steps=STEPS, model_id=MODEL_ID, revision=MODEL_REVISION)
generation['data_preparation'] = preparation
(OUTPUT_DIR/'generation_report.json').write_text(json.dumps(generation,ensure_ascii=False,indent=2),encoding='utf-8')
(OUTPUT_DIR/'git_commit.txt').write_text(commit,encoding='utf-8')
import matplotlib.pyplot as plt
pages=generation['pages']
fig,axes=plt.subplots(math.ceil(len(pages)/3),3,figsize=(15,5*math.ceil(len(pages)/3)),squeeze=False)
for ax in axes.flat: ax.axis('off')
for ax,page in zip(axes.flat,pages):
    with Image.open(page['input']) as image: ax.imshow(image.convert('RGB'))
    ax.set_title(page['label'])
plt.tight_layout(); plt.show()
print('전체 이미지와 고정 좌표 글자 crop:',OUTPUT_DIR)
print('생성 오류가 늘었는지는 원본과 잘라낸 결과를 확인하세요. 5번 셀에서 글자별로 평가합니다.')

In [ ]:
# 5번 셀 전체 교체: 기존 점수 + 추가 잉크 감점 위치 확인
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm

# 같은 생성 폴더의 점수 결과가 있으면 재사용합니다.
RECOMPUTE = False
same_run = False
if 'result' in globals():
    entries = result.get('extraction', {}).get('results', [])
    if entries:
        same_run = Path(entries[0]['npz']).resolve().parents[1] == Path(OUTPUT_DIR).resolve()
if RECOMPUTE or not same_run:
    result, scores = score_pilot(OUTPUT_DIR/'generation_report.json', CHECKPOINT,
        score_config=SCORE_CONFIG, source_config=SOURCE_CONFIG, show=True)
display(scores)

for sample, measured in zip(result['extraction']['results'], result['indicator']['results']):
    with np.load(measured['measurement_npz'], allow_pickle=False) as data:
        ink, extra, support = [data[key].copy() for key in ('image', 'extra_map', 'support')]
    fig, axes = plt.subplots(1, 4, figsize=(19, 4))
    axes[0].imshow(ink, cmap='gray_r', vmin=0, vmax=1)
    if support.min() < .5 < support.max():
        axes[0].contour(support, levels=[.5], colors=['cyan'], linewidths=.8)
    axes[0].set_title('Scored input + allowed region')
    heat = axes[1].imshow(extra, cmap='magma', norm=PowerNorm(gamma=.35, vmin=0, vmax=1))
    fig.colorbar(heat, ax=axes[1], fraction=.045)
    axes[1].set_title('Extra ink: enhanced faint values')
    axes[2].imshow(ink, cmap='gray_r', vmin=0, vmax=1)
    axes[2].imshow(np.ones((*ink.shape, 3))*[1, 0, 0], alpha=np.clip(extra/.08, 0, .85))
    axes[2].set_title('Extra overlay: display amplified')
    bins = [0, .02, .1, .5, 1.000001]
    mass = np.array([extra[(ink >= a) & (ink < b)].sum() for a, b in zip(bins[:-1], bins[1:])])
    shares = 100*mass/max(float(extra.sum()), 1e-12)
    axes[3].bar(['0-.02', '.02-.10', '.10-.50', '.50-1'], shares)
    axes[3].set(xlabel='Input ink intensity', ylabel='Share of extra-ink mass (%)', ylim=(0, 100))
    for ax in axes[:3]:
        ax.axis('off')
    fig.suptitle(f"{sample.get('label', '')} | score={measured['structural_score']:.1f} | "
                 f"before extra={measured['score_before_extra']:.1f} | extra fraction={100*measured['extra']:.2f}%")
    fig.tight_layout()
    path = Path(result['save_dir'])/f"extra_diagnosis_{measured['sample']:04d}.png"
    fig.savefig(path, dpi=140)
    plt.show()
    plt.close(fig)

print('빨강은 추가 잉크로 계산된 위치입니다. 옅은 성분을 보려고 표시 대비만 높였으며 점수는 바꾸지 않았습니다.')
print('막대의 왼쪽에 몰리면 옅은 픽셀의 영향, 오른쪽에 몰리면 진한 잉크의 영향을 확인하세요.')
print('밝기 구간만으로 배경과 실제 추가 획을 자동 판정하지는 않습니다.')
print('진단 그림 저장:', result['save_dir'])
from stroke_model.source_quality_v5 import compare_quality_v5
QUALITY_CONFIG = {}  # 실험 기본값. 원래 점수는 보존하고 새 비교 열을 추가합니다.
quality_report, quality_scores = compare_quality_v5(result, config=QUALITY_CONFIG, show=True)
display(quality_scores[['label','raw_source_score','foreground_structural_score',
    'contour_excess','contour_score_cap','experimental_score']])
print('새 점수는 실험용입니다. 정상 글꼴 차이와 실제 돌출·패임을 구분해 그림으로 확인하세요.')
print('윤곽 항은 현재 진단용이며, 역확산 guidance에 바로 미분해서 쓸 수 있는 구현은 아닙니다.')
from google.colab import files
download = widgets.Button(description='기존 진단 ZIP 다운로드')
debug_path = result['debug_zip']
download.on_click(lambda _, path=debug_path: files.download(path))
display(download)